# Módulo 5 · Notebook 1 — Tools para Agentes

Antes de construir agentes completos, necesitamos entender su bloque básico: las **tools**.

Una tool es una función que el modelo puede decidir usar cuando necesita:
- consultar datos externos,
- hacer cálculos,
- ejecutar lógica determinística.

En este notebook vas a:
1. Crear tools con `@tool`.
2. Probarlas directamente.
3. Ver cómo un LLM decide llamar tools (tool-calling).

## 1. Setup

In [ ]:
import warnings
from pathlib import Path
import pandas as pd
from langchain_core.tools import tool
from langchain_ollama import ChatOllama

warnings.filterwarnings("ignore")

# Usar rutas relativas robustas
BASE_DIR = Path("../../").resolve()
CSV_PATH = BASE_DIR / "data" / "table" / "WorldCupMatches.csv"

# Cargar el DataFrame real
df = pd.read_csv(CSV_PATH)

# Limpieza básica para cálculos y filtros
df["HomeTeamGoals"] = pd.to_numeric(df["HomeTeamGoals"], errors="coerce").fillna(0)
df["AwayTeamGoals"] = pd.to_numeric(df["AwayTeamGoals"], errors="coerce").fillna(0)
df["Stage"] = df["Stage"].astype(str).str.strip()

llm = ChatOllama(model="llama3.2:3b", temperature=0)
print(f"✅ Entorno y DataFrame listos. Filas cargadas: {len(df)}")

c:\_dev\IA\AI-course\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Entorno y DataFrame listos. Filas cargadas: 836


In [4]:
@tool
def campeon_del_mundial(anio: int) -> str:
    """Devuelve el campeón del Mundial para un año específico."""
    final = df[(df['Year'] == anio) & (df['Stage'] == 'Final')]
    
    if final.empty:
        return f"No se encontró un partido con Stage='Final' para el año {anio}."
        
    final = final.iloc[-1] 
    
    if final['HomeTeamGoals'] > final['AwayTeamGoals']:
        return str(final['HomeTeamName'])
    elif final['AwayTeamGoals'] > final['HomeTeamGoals']:
        return str(final['AwayTeamName'])
    else:
        return f"{final['HomeTeamName']} / {final['AwayTeamName']} (Decisión por penales/empate)"

@tool
def promedio_goles(inicio: int, fin: int) -> str:
    """Calcula el promedio de goles por mundial entre dos mundiales (años incluidos)."""
    df_rango = df[(df['Year'] >= inicio) & (df['Year'] <= fin)]
    
    if df_rango.empty:
        return "No hay mundiales en ese rango."
        
    df_rango = df_rango.assign(TotalMatchGoals=df_rango['HomeTeamGoals'] + df_rango['AwayTeamGoals'])
    goles_por_año = df_rango.groupby('Year')['TotalMatchGoals'].sum()
    promedio = goles_por_año.mean()
    
    return f"Promedio de goles por mundial entre {inicio} y {fin}: {promedio:.2f}"

@tool
def listar_campeones() -> str:
    """Lista los campeones de los mundiales disponibles en el dataset."""
    anios = sorted(df['Year'].dropna().unique())
    campeones_lista = []
    
    for anio in anios:
        final = df[(df['Year'] == anio) & (df['Stage'] == 'Final')]
        if not final.empty:
            final = final.iloc[-1]
            if final['HomeTeamGoals'] > final['AwayTeamGoals']:
                ganador = final['HomeTeamName']
            elif final['AwayTeamGoals'] > final['HomeTeamGoals']:
                ganador = final['AwayTeamName']
            else:
                ganador = str(final.get('WinConditions', f"{final['HomeTeamName']} o {final['AwayTeamName']}"))
                
            campeones_lista.append(f"{int(anio)}: {ganador}")
            
    return "\n".join(campeones_lista)

tools = [campeon_del_mundial, promedio_goles, listar_campeones]
print("Tools creadas:", [t.name for t in tools])

Tools creadas: ['campeon_del_mundial', 'promedio_goles', 'listar_campeones']


In [5]:
print("Campeón 1986:", campeon_del_mundial.invoke({"anio": 1986}))
print(promedio_goles.invoke({"inicio": 1990, "fin": 2010}))
print("\nLista de campeones:")
print(listar_campeones.invoke({}))

Campeón 1986: Argentina
Promedio de goles por mundial entre 1990 y 2010: 146.67

Lista de campeones:
1930: Uruguay
1934: Italy
1938: Italy
1954: Germany FR
1958: Brazil
1962: Brazil
1966: England
1970: Brazil
1974: Germany FR
1978: Argentina
1982: Italy
1986: Argentina
1990: Germany FR
1994: Brazil win on penalties (3 - 2) 
1998: France
2002: Brazil
2006: Italy win on penalties (5 - 3) 
2010: Spain
2014: Germany


## 2. Definir tools

Buenas prácticas:
- Nombre claro de la función.
- Docstring corto y preciso (el modelo la usa para decidir cuándo llamar la tool).
- Entrada/salida simple y tipada.

In [6]:
@tool
def campeon_del_mundial(anio: int) -> str:
    """Devuelve el campeón del Mundial para un año específico."""
    # Ahora usamos == 'Final' para ignorar Quarter-finals y Semi-finals
    final = df[(df['Year'] == anio) & (df['Stage'] == 'Final')]
    
    if final.empty:
        return f"No se encontró un partido con Stage='Final' explícito para el año {anio}."
        
    final = final.iloc[-1] # Tomamos el último registro por si hay duplicados
    
    if final['HomeTeamGoals'] > final['AwayTeamGoals']:
        return str(final['HomeTeamName'])
    elif final['AwayTeamGoals'] > final['HomeTeamGoals']:
        return str(final['AwayTeamName'])
    else:
        return f"{final['HomeTeamName']} / {final['AwayTeamName']} (Revisa WinConditions)"


@tool
def promedio_goles(inicio: int, fin: int) -> str:
    """Calcula el promedio de goles por mundial entre dos mundiales (años incluidos)."""
    df_rango = df[(df['Year'] >= inicio) & (df['Year'] <= fin)]
    
    if df_rango.empty:
        return "No hay mundiales en ese rango."
        
    df_rango = df_rango.assign(TotalMatchGoals=df_rango['HomeTeamGoals'] + df_rango['AwayTeamGoals'])
    
    # Agrupamos por año y promediamos
    goles_por_año = df_rango.groupby('Year')['TotalMatchGoals'].sum()
    promedio = goles_por_año.mean()
    
    return f"Promedio de goles por mundial entre {inicio} y {fin}: {promedio:.2f}"


@tool
def listar_campeones() -> str:
    """Lista los campeones de los mundiales disponibles en el dataset."""
    anios = sorted(df['Year'].dropna().unique())
    campeones_lista = []
    
    for anio in anios:
        final = df[(df['Year'] == anio) & (df['Stage'] == 'Final')]
        if not final.empty:
            final = final.iloc[-1]
            if final['HomeTeamGoals'] > final['AwayTeamGoals']:
                ganador = final['HomeTeamName']
            elif final['AwayTeamGoals'] > final['HomeTeamGoals']:
                ganador = final['AwayTeamName']
            else:
                ganador = str(final.get('WinConditions', f"{final['HomeTeamName']} o {final['AwayTeamName']}"))
                
            campeones_lista.append(f"{int(anio)}: {ganador}")
            
    return "\n".join(campeones_lista)

# --- FIN DE LAS TOOLS ---

tools = [campeon_del_mundial, promedio_goles, listar_campeones]
print("✅ Tools creadas con Pandas:", [t.name for t in tools])

print("\nPrueba 1 (Campeón 1986):", campeon_del_mundial.invoke({"anio": 1986}))
print("Prueba 2 (Promedio):", promedio_goles.invoke({"inicio": 1990, "fin": 2010}))

✅ Tools creadas con Pandas: ['campeon_del_mundial', 'promedio_goles', 'listar_campeones']

Prueba 1 (Campeón 1986): Argentina
Prueba 2 (Promedio): Promedio de goles por mundial entre 1990 y 2010: 146.67


## 3. Probar tools directamente

In [7]:
print(campeon_del_mundial.invoke({"anio": 1998}))
print(promedio_goles.invoke({"inicio": 1990, "fin": 2010}))
print(listar_campeones.invoke({})[:180], "...")

France
Promedio de goles por mundial entre 1990 y 2010: 146.67
1930: Uruguay
1934: Italy
1938: Italy
1954: Germany FR
1958: Brazil
1962: Brazil
1966: England
1970: Brazil
1974: Germany FR
1978: Argentina
1982: Italy
1986: Argentina
1990: Germa ...


## 4. Tool-calling con el LLM

Aquí no hay agente completo todavía. Solo dejamos que el LLM decida si necesita una tool.

In [8]:
llm_with_tools = llm.bind_tools(tools)

question = "¿Quién ganó el mundial de 1986?"
ai_msg = llm_with_tools.invoke(question)

print("Contenido del modelo:", ai_msg.content)
print("Tool calls:", ai_msg.tool_calls)

if ai_msg.tool_calls:
    # Ejecutamos manualmente la primera llamada para ver el ciclo básico
    call = ai_msg.tool_calls[0]
    selected_tool = {
        "campeon_del_mundial": campeon_del_mundial,
        "promedio_goles": promedio_goles,
        "listar_campeones": listar_campeones,
    }[call["name"]]
    tool_result = selected_tool.invoke(call["args"])
    print("Resultado de tool:", tool_result)
else:
    print("El modelo respondió sin usar tools.")

Contenido del modelo: 
Tool calls: [{'name': 'campeon_del_mundial', 'args': {'anio': 1986}, 'id': '1cf4ae9f-ab94-4fa4-b196-458c4ad254e4', 'type': 'tool_call'}]
Resultado de tool: Argentina


## 5. Resumen

- Las tools son funciones con interfaz clara para que el modelo las use.
- `@tool` convierte una función Python en una herramienta compatible con LangChain.
- `bind_tools` habilita la decisión de llamar herramientas.
- En el siguiente notebook automatizamos el ciclo completo con un agente ReAct.